# Fluxo de trabalho: DEM → HAND (Height Above Nearest Drainage)
# mapas de suscetibilidade/probabilidade de inundação (abordagem HAND)

Este notebook calcula **HAND** a partir de um **Modelo Digital de Elevação (DEM)** usando **WhiteboxTools** e, em seguida (opcionalmente), deriva:

- um **mapa de suscetibilidade** (classes de risco) baseado em limiares de HAND; e
- um **mapa de probabilidade de inundação** (heurístico) a partir de uma função monótona de HAND (*seção HANN*).

> **Ideia física (intuição):** o HAND mede, para cada pixel, quantos **metros** ele está acima da **drenagem mais próxima** (seguindo a conectividade hidrológica). Em geral, **HAND baixo ⇒ maior propensão a inundar**, porque o terreno está mais “perto” do nível do canal.

### Entradas
- **DEM** (GeoTIFF): elevação do terreno.
- Parâmetros da análise hidrológica (por exemplo, limiar de acumulação para extrair a rede de drenagem).

### Saídas principais
- `*_hand.tif`: raster HAND (mesmo sistema de referência/extensão do DEM).

### Saídas opcionais (pós-processamento, sem alterar o pipeline)
- `*_flood_risk_classes.tif`: classes discretas de risco por limiares.
- `*_flood_probability.tif`: probabilidade (0–1) derivada de HAND.


## Recomendações de qualidade dos dados (importante)
- **CRS projetado (metros):** se o seu DEM estiver em lat/long (graus), reprojete para um CRS projetado para que as magnitudes de distância/área sejam coerentes.
- **NoData consistente:** valores NoData incorretos contaminam direção/acumulação de fluxo.
- **Resolução:** a resolução do DEM define o “detalhe” hidrológico; DEMs muito grosseiros suavizam canais e planícies.

## 1) Requisitos e instalação

### Bibliotecas
- **`whitebox` / `WhiteboxTools`**: executa ferramentas de análise hidrológica e geomorfométrica (fill/breach depressions, fluxo D8, acumulação, extração de drenagens, HAND).

## Glossário rápido de termos do pipeline

- **DEM (Digital Elevation Model)**: raster onde cada pixel armazena elevação (idealmente em metros).
- **Depressão / sink**: conjunto de células onde o fluxo fica preso sem saída para uma célula mais baixa (pode ser real ou um artefato).
- **Fill depressions**: remove sinks elevando o terreno no mínimo necessário para garantir saída.
- **Breach depressions least cost**: remove sinks criando um “corte” até a saída minimizando a modificação do relevo.
- **DEM conditioning**: conjunto de passos para que o DEM seja hidrologicamente consistente (sem artefatos que quebrem a conectividade do fluxo).
- **D8 pointer (direção de fluxo)**: raster que codifica para qual dos 8 vizinhos escoa cada célula.
- **D8 flow accumulation (acumulação)**: raster que quantifica a contribuição a montante para cada célula (em #células ou área).
- **Stream extraction (extração de drenagens)**: define a rede de canais aplicando um limiar sobre a acumulação.


In [ ]:
import os

# Esta célula é opcional para execução no Google Colab.
# Em execução local, o notebook usa PROJECT_ROOT e caminhos relativos.
try:
    from google.colab import drive
    if not os.path.exists("/content/drive"):
        drive.mount("/content/drive")
    print("Google Colab disponível: Drive pronto para uso opcional.")
except Exception as e:
    print("Execução local detectada; Google Colab é opcional.")
    print("Nesta execução, o notebook usa PROJECT_ROOT e caminhos relativos.")
    print("Detalhe:", e)


In [ ]:
# ============================================================
# Configuração de entrada (AJUSTE AQUI)
# ============================================================

projectName = "HAND"
numProjetos = 1
profundidadeMax = 8

# ============================================================
# Configuração de ambiente local / Google Drive
# ============================================================

import os
from pathlib import Path

def encontrar_pastas_por_nome(raiz: Path, nome_pasta: str, profundidade_max: int, limite: int = 1):
    """
    Procura pastas com nome 'nome_pasta' dentro de 'raiz', varrendo recursivamente até 'profundidade_max'.
    Retorna uma lista de Paths (tamanho no máximo 'limite').
    """
    resultados = []
    fila = [(raiz, 0)]

    while fila and len(resultados) < limite:
        atual, profundidade = fila.pop(0)

        if profundidade >= profundidade_max:
            continue

        try:
            for item in atual.iterdir():
                if item.is_dir():
                    if item.name == nome_pasta:
                        resultados.append(item)
                        if len(resultados) >= limite:
                            break
                    fila.append((item, profundidade + 1))
        except PermissionError:
            continue

    return resultados

def encontrar_raiz_local(inicio: Path) -> Path | None:
    marcadores = [
        Path("notebooks") / "blumenau" / "hand_whitebox_integrado_ANA_IBGE_BLUMENAU.ipynb",
        Path("notebooks") / "base" / "hand_whitebox_integrado_ANA_IBGE_POA.ipynb",
        Path("data") / "raw" / "BR_Municipios_2023" / "BR_Municipios_2023.shp",
        Path("outputs") / "blumenau",
    ]

    candidatos = [inicio, *inicio.parents]
    for candidato in candidatos:
        if any((candidato / marcador).exists() for marcador in marcadores):
            return candidato
    return None

if not isinstance(projectName, str) or not projectName.strip():
    raise ValueError("A variável 'projectName' deve ser uma string não vazia.")

if not isinstance(numProjetos, int) or numProjetos < 1:
    raise ValueError("A variável 'numProjetos' deve ser um inteiro >= 1.")

if not isinstance(profundidadeMax, int) or profundidadeMax < 1:
    raise ValueError("A variável 'profundidadeMax' deve ser um inteiro >= 1.")

profundidadeMax = min(profundidadeMax, 8)

diretorioEntrada = encontrar_raiz_local(Path.cwd())

if diretorioEntrada is not None:
    print("Ambiente local detectado.")
else:
    # Esta busca no /content/drive é apenas compatibilidade opcional com Google Colab.
    # Em execução local, o notebook usa PROJECT_ROOT e caminhos relativos.
    raizDrive = Path("/content/drive/MyDrive")
    if raizDrive.exists():
        pastasEncontradas = encontrar_pastas_por_nome(
            raiz=raizDrive,
            nome_pasta=projectName.strip(),
            profundidade_max=profundidadeMax,
            limite=numProjetos,
        )
        if pastasEncontradas:
            diretorioEntrada = pastasEncontradas[0]
            print("Google Drive detectado.")
            print("Pasta(s) encontrada(s):")
            for i, p in enumerate(pastasEncontradas, start=1):
                print(f"  {i}. {p}")

if diretorioEntrada is None:
    raise FileNotFoundError(
        "Não foi possível localizar a pasta do projeto nem no ambiente local nem no Google Drive."
    )

PROJECT_ROOT = Path(diretorioEntrada).resolve()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
BLUMENAU_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "blumenau"

os.chdir(PROJECT_ROOT)

print("\nDiretório de entrada definido como:")
print(PROJECT_ROOT)
print("\nDiretório de trabalho atual:")
print(Path.cwd())
print("\nDados brutos em:")
print(RAW_DATA_DIR)
print("\nSaídas de Blumenau em:")
print(BLUMENAU_OUTPUT_DIR)


In [ ]:
# ============================================================
# Instalação de dependências a partir do diretório do projeto
# ============================================================

import os
import sys
import subprocess
from pathlib import Path

if "diretorioEntrada" not in globals():
    raise NameError(
        "A variável 'diretorioEntrada' não foi encontrada. "
        "Execute primeiro o bloco que localiza a pasta do projeto."
    )

diretorioEntrada = Path(diretorioEntrada)
if not diretorioEntrada.exists():
    raise FileNotFoundError(f"O diretório de entrada não existe: {diretorioEntrada}")

print(f"Diretório do projeto: {diretorioEntrada}")

def run_cmd(cmd, cwd=None):
    print("\nExecutando:", " ".join(cmd))
    res = subprocess.run(cmd, cwd=str(cwd) if cwd else None, text=True, capture_output=True)
    if res.stdout:
        print(res.stdout)
    if res.returncode != 0:
        if res.stderr:
            print(res.stderr)
        raise RuntimeError(f"Falha ao executar o comando: {' '.join(cmd)}")
    return res

def pip_install_requirements(req_file: Path):
    run_cmd([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])
    run_cmd([sys.executable, "-m", "pip", "install", "-r", str(req_file)])

def pip_install_packages(packages):
    run_cmd([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])
    run_cmd([sys.executable, "-m", "pip", "install", *packages])

required_modules = {
    "geopandas": "geopandas",
    "rasterio": "rasterio",
    "rioxarray": "rioxarray",
    "xarray": "xarray",
    "contextily": "contextily",
    "whitebox": "whitebox",
    "shapely": "shapely",
    "fiona": "fiona",
    "planetary_computer": "planetary-computer",
    "pystac_client": "pystac-client",
}

missing_packages = []
for module_name, package_name in required_modules.items():
    try:
        __import__(module_name)
    except Exception:
        missing_packages.append(package_name)

if not missing_packages:
    print("Dependências principais já estão disponíveis. Pulando instalação.")
else:
    candidatos = [
        "requirements.txt",
        "requirements-dev.txt",
        "requirements/requirements.txt",
        "requirements/base.txt",
    ]

    req_encontrado = None
    for rel in candidatos:
        p = diretorioEntrada / rel
        if p.exists() and p.is_file():
            req_encontrado = p
            break

    if req_encontrado is None:
        for p in sorted(diretorioEntrada.glob("requirements*.txt")):
            if p.is_file():
                req_encontrado = p
                break

    if req_encontrado:
        print(f"Arquivo de requisitos encontrado: {req_encontrado}")
        pip_install_requirements(req_encontrado)
        print("Instalação finalizada com sucesso via requirements.")
    else:
        print("Nenhum requirements.txt encontrado. Instalando apenas os pacotes ausentes do notebook...")
        pip_install_packages(sorted(set(missing_packages)))
        print("Instalação finalizada com sucesso via lista mínima de pacotes.")

In [ ]:
# =============================
# Bibliotecas padrão (stdlib)
# =============================
import os
import sys
import zipfile
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str((Path.cwd() / ".matplotlib").resolve()))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

# =============================
import contextily as cx
import fiona
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import planetary_computer
import rasterio
import requests
import rioxarray as rxr
import xarray as xr
from pystac_client import Client
from rasterio.mask import mask
from shapely.ops import unary_union

try:
    from whitebox.whitebox_tools import WhiteboxTools
except Exception as e:
    print(
        f"Erro {e}: pacote 'whitebox' não encontrado. Instale com: pip install whitebox",
        file=sys.stderr,
    )
    raise

## 2) Funções auxiliares

Estas funções fazem duas coisas:

1. **Validação de entradas numéricas** (por exemplo, garantir que o limiar de acumulação seja positivo).
2. Manter um comportamento equivalente ao script original, mas com erros mais explícitos caso algo venha errado.

- `positive_int`: força inteiro positivo (útil para `threshold_cells`).
- `positive_float`: força float positivo (útil se converter área ou tamanho de célula).



In [ ]:
import argparse

def positive_int(value):
    """Converte para int e verifica se é > 0."""
    ivalue = int(value)
    if ivalue <= 0:
        raise argparse.ArgumentTypeError("El valor debe ser un entero positivo")
    return ivalue

def positive_float(value):
    """Converte para float e verifica se é > 0."""
    fvalue = float(value)
    if fvalue <= 0:
        raise argparse.ArgumentTypeError("El valor debe ser un número positivo")
    return fvalue


## 3) Entradas (inputs) e saídas (outputs)

Nesta célula você define os parâmetros do fluxo.

### Parâmetros-chave
- `dem`: caminho para o DEM (GeoTIFF).
- `outdir`: pasta de saída.
- `prefix`: prefixo para os arquivos gerados.
- `breach`:
  - `False`: usa `fill_depressions` (preenche depressões).
  - `True`: usa `breach_depressions_least_cost` (rompe depressões com custo mínimo).
- `threshold_cells`: limiar (em número de células) para extração da rede de drenagem.
  - quanto maior, menos canais; quanto menor, mais canais.

### Produtos gerados (principais)
- `*_dem_filled.tif`: DEM condicionado (preenchido/rompido).
- `*_d8_pointer.tif`: direção de fluxo D8.
- `*_d8_accum.tif`: acumulação de fluxo D8.
- `*_streams.tif`: rede de drenagem (binária).
- `*_hand.tif`: HAND (metros acima da drenagem mais próxima).


### Influência de `threshold_cells` no resultado (e no tempo)

Do ponto de vista hidrológico, `threshold_cells` controla o tamanho mínimo de bacia (em células) para formar um canal.

- Se **aumentar** `threshold_cells`: você obtém **menos** canais; o HAND tende a referenciar drenagens principais.
- Se **reduzir** `threshold_cells`: você obtém **mais** canais; o HAND fica mais “local” (mais drenagens pequenas).

Em termos computacionais, isso influencia principalmente `extract_streams` e (dependendo do tamanho do DEM) pode alterar tempos e tamanho de arquivos.



## 3) Insumos oficiais (ANA + IBGE) para Blumenau: DEM e Áreas de Contribuição (Ottobacias)

Nesta seção, *preparamos os insumos* que alimentarão o pipeline HAND:

1. **Limite municipal de Blumenau (IBGE)**  
   - Usa-se a *Malha Municipal* do IBGE e filtra-se o município **Blumenau (SC)**.

2. **Áreas de contribuição hidrográfica (ANA / BHO)**  
   - Consulta-se o serviço oficial (ArcGIS REST) de **Área de Drenagem - BHO**, que representa as **ottobacias** (áreas de contribuição) associadas à rede hidrográfica.
   - Selecionam-se as ottobacias que **intersectam** o polígono municipal de Blumenau.
   - Permite-se filtrar por **nível de contribuição** (campo `nunivotto`, nível Otto).

3. **DEM (Modelo Digital de Elevação) recortado pelo contorno de contribuição**  
   - Faz-se o download de um DEM (preferencialmente **ANADEM** se estiver disponível; caso contrário, usa-se o **Copernicus DEM GLO-30** via Planetary Computer).
   - Recorta-se o DEM pelo **contorno (união) das ottobacias selecionadas** (e opcionalmente também pelo limite municipal).
   - O resultado é o DEM “ajustado” usado como entrada do HAND.

> **Saída:** todos os produtos são salvos dentro de `outdir` (a pasta de saída do notebook).


# Fluxo de trabalho: DEM

In [ ]:
try:
    # =========================================================
    # [3.1] Baixar limites municipais (IBGE) e ottobacias (ANA/BHO),
    #       depois selecionar as áreas de contribuição que intersectam Blumenau
    # =========================================================
    #
    # Produtos (em outdir):
    #   - blumenau_boundary.gpkg (+ .shp opcional)
    #   - ottobacias_blumenau_raw.gpkg  (todas as que intersectam)
    #   - ottobacias_blumenau_by_level.gpkg (filtradas por nível Otto)
    #   - ottobacias_blumenau_union.gpkg (contorno final para recorte do DEM)

    external_errors = globals().setdefault("external_errors", [])
    current_source = "IBGE"

    PROJECT_ROOT = Path(globals().get("PROJECT_ROOT", Path.cwd())).resolve()
    RAW_DATA_DIR = Path(globals().get("RAW_DATA_DIR", PROJECT_ROOT / "data" / "raw"))
    outdir = Path(globals().get("BLUMENAU_OUTPUT_DIR", PROJECT_ROOT / "outputs" / "blumenau"))
    outdir.mkdir(parents=True, exist_ok=True)

    IBGE_MUNICIPIOS_YEAR = 2023
    CONTRIBUTION_LEVELS = [5]
    CLIP_TO_MUNICIPALITY = False

    ibge_zip = RAW_DATA_DIR / f"BR_Municipios_{IBGE_MUNICIPIOS_YEAR}.zip"
    ibge_extract_dir = RAW_DATA_DIR / f"BR_Municipios_{IBGE_MUNICIPIOS_YEAR}"
    local_ibge_shp = RAW_DATA_DIR / f"BR_Municipios_{IBGE_MUNICIPIOS_YEAR}" / f"BR_Municipios_{IBGE_MUNICIPIOS_YEAR}.shp"

    blumenau_gpkg = outdir / "blumenau_boundary.gpkg"
    ottos_raw_gpkg = outdir / "ottobacias_blumenau_raw.gpkg"
    ottos_lvl_gpkg = outdir / "ottobacias_blumenau_by_level.gpkg"
    ottos_union_gpkg = outdir / "ottobacias_blumenau_union.gpkg"

    if local_ibge_shp.exists():
        ibge_shp = local_ibge_shp
        print(f"Usando shapefile local do IBGE: {ibge_shp}")
    else:
        ibge_url_br = (
            "https://geoftp.ibge.gov.br/organizacao_do_territorio/malhas_territoriais/"
            f"malhas_municipais/municipio_{IBGE_MUNICIPIOS_YEAR}/Brasil/"
            f"BR_Municipios_{IBGE_MUNICIPIOS_YEAR}.zip"
        )
        ibge_url_sc = (
            "https://geoftp.ibge.gov.br/organizacao_do_territorio/malhas_territoriais/"
            f"malhas_municipais/municipio_{IBGE_MUNICIPIOS_YEAR}/UFs/SC/"
            f"SC_Municipios_{IBGE_MUNICIPIOS_YEAR}.zip"
        )

        def download(url: str, dst: Path, timeout=180):
            response = requests.get(url, timeout=timeout)
            response.raise_for_status()
            dst.write_bytes(response.content)

        if not ibge_zip.exists():
            try:
                print(f"Baixando malha municipal do IBGE (BR) {IBGE_MUNICIPIOS_YEAR}...")
                download(ibge_url_br, ibge_zip)
                print("OK (BR).")
            except Exception as e:
                external_errors.append(f"[IBGE] {type(e).__name__}: {e}")
                print(f"Falhou BR ({e}). Tentando SC...")
                ibge_zip = RAW_DATA_DIR / f"SC_Municipios_{IBGE_MUNICIPIOS_YEAR}.zip"
                current_source = "IBGE"
                download(ibge_url_sc, ibge_zip)
                print("OK (SC).")
        else:
            print("ZIP do IBGE já existe. OK.")

        if not ibge_extract_dir.exists():
            print("Extraindo ZIP do IBGE...")
            ibge_extract_dir.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(ibge_zip, "r") as zf:
                zf.extractall(ibge_extract_dir)
            print("Extraído.")
        else:
            print("ZIP do IBGE já extraído. OK.")

        shps = list(ibge_extract_dir.rglob("*.shp"))
        if not shps:
            raise FileNotFoundError(f"Não foi encontrado .shp dentro de {ibge_extract_dir}")
        ibge_shp = shps[0]
        print(f"Shapefile do IBGE encontrado: {ibge_shp.name}")

    mun = gpd.read_file(ibge_shp)

    required_cols = {"NM_MUN", "CD_UF"}
    missing_cols = required_cols.difference(mun.columns)
    if missing_cols:
        raise KeyError(
            f"Não encontrei as colunas {sorted(missing_cols)}. Colunas disponíveis: {mun.columns.tolist()}"
        )

    blumenau = mun[
        (mun["NM_MUN"].astype(str).str.strip().str.upper() == "BLUMENAU")
        & (mun["CD_UF"].astype(str) == "42")
    ].copy()

    if blumenau.empty:
        raise ValueError("Blumenau (SC) não encontrado na malha municipal.")

    blumenau = blumenau.to_crs(4326)
    blumenau.to_file(blumenau_gpkg, layer="blumenau", driver="GPKG")

    print(f"Limite municipal salvo em: {blumenau_gpkg}")

    current_source = "SNIRH/ANA"

    blumenau_3857 = blumenau.to_crs(3857)
    xmin, ymin, xmax, ymax = blumenau_3857.total_bounds

    snirh_url = "https://www.snirh.gov.br/arcgis/rest/services/otto_bho_2017/MapServer/0/query"
    params = {
        "f": "geojson",
        "where": "1=1",
        "geometry": f"{xmin},{ymin},{xmax},{ymax}",
        "geometryType": "esriGeometryEnvelope",
        "inSR": 3857,
        "spatialRel": "esriSpatialRelIntersects",
        "outFields": "*",
        "returnGeometry": "true",
        "outSR": 3857,
    }

    resp = requests.get(snirh_url, params=params, timeout=180)
    resp.raise_for_status()
    gj = resp.json()

    feats = gj.get("features", [])
    if not feats:
        raise RuntimeError("A consulta SNIRH/ANA não retornou ottobacias para o envelope de Blumenau.")

    ottos_3857 = gpd.GeoDataFrame.from_features(feats, crs="EPSG:3857")
    ottos_3857 = ottos_3857[ottos_3857.intersects(blumenau_3857.geometry.iloc[0])].copy()

    if ottos_3857.empty:
        raise RuntimeError("Nenhuma ottobacia intersecta exatamente Blumenau após o filtro espacial.")

    ottos_3857.to_file(ottos_raw_gpkg, layer="ottos_raw", driver="GPKG")
    print(f"Ottobacias intersectantes salvas em: {ottos_raw_gpkg}")

    level_cols = [c for c in ottos_3857.columns if "nivel" in c.lower() or "otto" in c.lower()]
    if not level_cols:
        ottos = ottos_3857.copy()
    else:
        candidate = None
        for col in level_cols:
            lower = col.lower()
            if "nivel" in lower and any(char.isdigit() for char in lower):
                candidate = col
                break
        if candidate is None:
            for col in level_cols:
                if "nivel" in col.lower():
                    candidate = col
                    break
        if candidate is None:
            candidate = level_cols[0]

        ottos = ottos_3857[ottos_3857[candidate].isin(CONTRIBUTION_LEVELS)].copy()
        if ottos.empty:
            print(
                f"Nenhuma ottobacia encontrada nos níveis {CONTRIBUTION_LEVELS} em '{candidate}'. "
                "Mantendo todas as ottobacias intersectantes."
            )
            ottos = ottos_3857.copy()

    ottos.to_file(ottos_lvl_gpkg, layer="ottos_by_level", driver="GPKG")
    print(f"Ottobacias por nível salvas em: {ottos_lvl_gpkg}")

    union_geom = ottos.unary_union
    if CLIP_TO_MUNICIPALITY:
        union_geom = union_geom.intersection(blumenau_3857.geometry.iloc[0])

    union_gdf = gpd.GeoDataFrame(
        {"name": ["contrib_union"]},
        geometry=[union_geom],
        crs="EPSG:3857",
    )
    union_gdf.to_file(ottos_union_gpkg, layer="contrib_union", driver="GPKG")
    print(f"Contorno final de contribuição salvo em: {ottos_union_gpkg}")

except Exception as e:
    external_errors = globals().setdefault("external_errors", [])
    external_errors.append(f"[{current_source}] {type(e).__name__}: {e}")
    print(f"Erro na etapa [3.1] ({current_source}): {type(e).__name__}: {e}")
    raise


In [ ]:
if "ottos" not in globals() or ottos is None or ottos.empty:
    print("Sem ottobacias disponíveis para plotar nesta etapa.")
else:
    if ottos.crs is None:
        raise ValueError("O GeoDataFrame 'ottos' não possui CRS definido.")

    ottos_3857 = ottos.to_crs(epsg=3857)

    fig, ax = plt.subplots(figsize=(10, 10))
    ottos_3857.plot(
        ax=ax,
        edgecolor="black",
        linewidth=0.8,
        alpha=0.6,
    )

    cx.add_basemap(
        ax,
        source=cx.providers.OpenStreetMap.Mapnik,
        attribution_size=6,
    )
    ax.set_axis_off()
    ax.set_title("Pegada hidrológica do território - Ottobacias BHO", fontsize=12)

    plt.tight_layout()
    plt.show()

In [ ]:
if "ottos" not in globals() or ottos is None or ottos.empty:
    print("Sem ottobacias disponíveis para plotar nesta etapa.")
else:
    if ottos.crs is None:
        raise ValueError("O GeoDataFrame 'ottos' não possui CRS definido.")

    ottos_3857 = ottos.to_crs(epsg=3857)

    fig, ax = plt.subplots(figsize=(10, 10))
    ottos_3857.plot(
        ax=ax,
        edgecolor="black",
        linewidth=0.8,
        alpha=0.6,
    )

    cx.add_basemap(
        ax,
        source=cx.providers.OpenStreetMap.Mapnik,
        attribution_size=6,
    )
    ax.set_axis_off()
    ax.set_title("Pegada hidrológica do território - Ottobacias BHO", fontsize=12)

    plt.tight_layout()
    plt.show()

In [ ]:
# =========================================================
# [3.2] Baixar o DEM e recortá-lo pelo contorno de contribuição
# =========================================================
# Este bloco tenta usar o ANADEM (ANA/UFRGS) se estiver disponível.
# Se o servidor de tiles bloquear o acesso, muda automaticamente para o Copernicus DEM (GLO-30)
# via Planetary Computer (Microsoft) como alternativa reprodutível.
#
# Produtos (em outdir):
#   - dem_source.tif               (DEM baixado)
#   - dem_contrib_clipped.tif      (DEM recortado pelo contorno de contribuição)
#   - (opcionais) imagens quicklook PNG

external_errors = globals().setdefault("external_errors", [])
current_source = "Pré-processamento DEM"

try:
    if "ottos_union_gpkg" not in globals():
        raise NameError("A variável 'ottos_union_gpkg' não foi definida na etapa [3.1].")

    if not Path(ottos_union_gpkg).exists():
        raise FileNotFoundError(
            f"Não encontrei o contorno de contribuição esperado em {ottos_union_gpkg}. "
            "Verifique a etapa [3.1] antes de continuar."
        )

    union_gdf = gpd.read_file(ottos_union_gpkg, layer="contrib_union")
    dem_download = outdir / "dem_source.tif"
    dem_clipped = outdir / "dem_contrib_clipped.tif"

    # Blumenau também está em área compatível com o tile 22J; se falhar, usar fallback Copernicus DEM.
    ANADEM_TILE = "22J"
    ANADEM_URL = f"https://metadados.snirh.gov.br/files/anadem_v1_tiles/anadem_v1_{ANADEM_TILE}.tif"

    def validate_raster(path: Path) -> bool:
        try:
            with rasterio.open(path) as src:
                _ = src.count, src.width, src.height, src.crs
            return True
        except Exception as e:
            print(f"Raster inválido em {path.name}: {type(e).__name__}: {e}")
            return False

    def download_with_resume(url, dst, max_retries=5, chunk_size=8 * 1024 * 1024):
        part = dst.with_suffix(dst.suffix + ".part")

        if dst.exists():
            print(f"Validando arquivo existente: {dst.name}")
            if validate_raster(dst):
                print("Arquivo existente válido. Reutilizando download anterior.")
                return True
            print("Arquivo existente corrompido. Removendo e baixando novamente.")
            dst.unlink()

        for attempt in range(1, max_retries + 1):
            try:
                headers = {"User-Agent": "Mozilla/5.0"}
                if part.exists() and part.stat().st_size > 0:
                    headers["Range"] = f"bytes={part.stat().st_size}-"
                    print(
                        f"Tentativa {attempt}/{max_retries}: retomando download de {part.stat().st_size} bytes..."
                    )
                else:
                    print(f"Tentativa {attempt}/{max_retries}: iniciando download completo...")

                with requests.get(url, headers=headers, stream=True, timeout=(30, 300)) as response:
                    if response.status_code not in (200, 206):
                        raise RuntimeError(f"HTTP {response.status_code} ao acessar {url}")

                    mode = "ab" if response.status_code == 206 and part.exists() else "wb"
                    if response.status_code == 200 and part.exists() and "Range" in headers:
                        print("Servidor reiniciou o download em vez de retomar; sobrescrevendo o .part.")
                        mode = "wb"

                    with open(part, mode) as f:
                        for chunk in response.iter_content(chunk_size=chunk_size):
                            if chunk:
                                f.write(chunk)

                if validate_raster(part):
                    part.rename(dst)
                    print(f"Download ANADEM concluído com sucesso: {dst.name}")
                    return True

                print("Arquivo parcial ainda não está íntegro após a tentativa; mantendo .part para retomada.")

            except Exception as e:
                print(f"Falha na tentativa {attempt}/{max_retries}: {type(e).__name__}: {e}")
                if attempt == max_retries:
                    external_errors.append(f"[ANADEM] {type(e).__name__}: {e}")

        if part.exists() and not validate_raster(part):
            print("Arquivo .part final inválido; removendo resíduo corrompido.")
            part.unlink()

        return False

    def download_copernicus_dem(dst: Path, bbox_4326):
        from rasterio.merge import merge

        catalog = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1")
        search = catalog.search(collections=["cop-dem-glo-30"], bbox=bbox_4326)
        items = list(search.items())
        if len(items) == 0:
            raise RuntimeError("Não foram encontrados tiles do Copernicus DEM para o bbox.")

        signed_items = [planetary_computer.sign(item) for item in items]
        sources = []
        try:
            for item in signed_items:
                href = item.assets["data"].href
                src = rasterio.open(href)
                sources.append(src)

            mosaic, out_transform = merge(sources, bounds=tuple(bbox_4326))
            meta = sources[0].meta.copy()
            meta.update(
                {
                    "driver": "GTiff",
                    "height": mosaic.shape[1],
                    "width": mosaic.shape[2],
                    "transform": out_transform,
                    "compress": "deflate",
                }
            )

            with rasterio.open(dst, "w", **meta) as out_ds:
                out_ds.write(mosaic)
        finally:
            for src in sources:
                src.close()

    current_source = "ANADEM"
    ok = download_with_resume(ANADEM_URL, dem_download, max_retries=5)

    if not ok:
        current_source = "Copernicus DEM"
        bbox_4326 = union_gdf.to_crs(4326).total_bounds.tolist()
        print("ANADEM não concluiu. Tentando fallback Copernicus DEM para bbox:", bbox_4326)
        try:
            download_copernicus_dem(dem_download, bbox_4326)
            if not validate_raster(dem_download):
                raise RuntimeError("O arquivo baixado via Copernicus DEM não passou na validação raster.")
        except Exception as e:
            external_errors.append(f"[Copernicus DEM] {type(e).__name__}: {e}")
            raise

    if not validate_raster(dem_download):
        raise RuntimeError(f"O raster DEM final não pôde ser validado: {dem_download}")

    with rasterio.open(dem_download) as src:
        union_dem_crs = union_gdf.to_crs(src.crs)
        geoms = [union_dem_crs.geometry.iloc[0].__geo_interface__]

        out_img, out_transform = mask(src, geoms, crop=True)
        out_meta = src.meta.copy()
        nodata_val = src.nodata if src.nodata is not None else -32768

        if out_meta.get("dtype") != out_img.dtype.name:
            out_meta["dtype"] = out_img.dtype.name

        out_meta.update(
            {
                "driver": "GTiff",
                "height": out_img.shape[1],
                "width": out_img.shape[2],
                "transform": out_transform,
                "nodata": nodata_val,
                "compress": "deflate",
            }
        )

        with rasterio.open(dem_clipped, "w", **out_meta) as dst:
            dst.write(out_img)

    if not validate_raster(dem_clipped):
        raise RuntimeError(f"O DEM recortado foi criado, mas falhou na validação: {dem_clipped}")

    dem = dem_clipped
    print(f"DEM recortado guardado: {dem_clipped.name}")
    print(f"Tamanho do dem_source.tif: {dem_download.stat().st_size} bytes")
    print(f"Tamanho do dem_contrib_clipped.tif: {dem_clipped.stat().st_size} bytes")
    print("Variável 'dem' atualizada para HAND:", dem)

except Exception as e:
    err_msg = f"[{current_source}] {type(e).__name__}: {e}"
    external_errors.append(err_msg)
    print(f"Erro no bloco DEM: {err_msg}")
    print("[i] O notebook adaptado foi preservado, mas o DEM de Blumenau não foi preparado nesta execução.")

In [ ]:
if "dem" not in globals():
    print("Visualização do DEM ignorada porque a variável 'dem' ainda não foi definida.")
else:
    dem_path = Path(dem)
    if not dem_path.exists():
        print(f"Visualização do DEM ignorada porque o arquivo não existe: {dem_path}")
    else:
        r = (
            rxr.open_rasterio(str(dem_path), masked=True)
              .squeeze()
              .rio.reproject(3857)
        )

        fig, ax = plt.subplots(figsize=(10, 10))

        r.plot(
            ax=ax,
            cmap="terrain",
            alpha=0.75,
            robust=True,
            add_colorbar=True,
        )

        cx.add_basemap(
            ax,
            source=cx.providers.Esri.WorldImagery,
            attribution_size=6,
        )

        ax.set_axis_off()
        ax.set_title(
            "Modelo Digital de Elevação sobre imagem de satélite",
            fontsize=14,
            weight="bold",
        )

        plt.tight_layout()
        plt.show()

# Fluxo de trabalho: HAND (Height Above Nearest Drainage)

In [ ]:
Path.cwd()

In [ ]:
# # ======================================================
# # Parâmetros (mesma lógica do script original)
# # ======================================================

PROJECT_ROOT = Path(globals().get("PROJECT_ROOT", diretorioEntrada if "diretorioEntrada" in globals() else Path.cwd())).resolve()
BLUMENAU_OUTPUT_DIR = Path(globals().get("BLUMENAU_OUTPUT_DIR", PROJECT_ROOT / "outputs" / "blumenau"))

dem_preferencial = BLUMENAU_OUTPUT_DIR / "dem_contrib_clipped.tif"

if "dem" in globals() and Path(dem).exists():
    dem = Path(dem)
elif dem_preferencial.exists():
    dem = dem_preferencial
else:
    dem = dem_preferencial

outdir = BLUMENAU_OUTPUT_DIR
outdir.mkdir(parents=True, exist_ok=True)

prefix = "blumenau_hand"

breach = False
threshold_cells = 100
cellsize_m = None
area_km2 = None
keep_intermediates = False

print("DEM:", dem)
print("Saída:", outdir)


## 4) Validação do DEM e cálculo do limiar de canais

Esta etapa garante que:

1. O arquivo de DEM realmente existe.
2. O limiar de extração de drenagens (`threshold_cells`) seja definido de forma coerente.

Regras:

* Se `threshold_cells` **não for None**, ele é validado como um inteiro positivo.
* Se `threshold_cells` for **None**, mas `cellsize_m` e `area_km2` estiverem definidos:

  * Converte-se a área para número de células:

    * `cells = (area_km2 * 1e6) / (cellsize_m²)`

> Isso permite especificar um limiar em termos hidrológicos (“canal a partir de bacias ≥ X km²”), o que costuma ser mais interpretável.

In [ ]:
if not Path(dem).exists():
    raise FileNotFoundError(
        f"DEM não encontrado: {dem}. "
        f"Erros externos registrados: {globals().get('external_errors', [])}"
    )

if threshold_cells is not None:
    threshold_cells = positive_int(threshold_cells)
elif (cellsize_m is not None) and (area_km2 is not None):
    cellsize_m = positive_float(cellsize_m)
    area_km2 = positive_float(area_km2)
    threshold_cells = int((area_km2 * 1_000_000.0) / (cellsize_m ** 2))
else:
    raise ValueError("Você deve definir threshold_cells, ou então (cellsize_m e area_km2).")

print("Limiar final (células):", threshold_cells)

## 5) Configuração do WhiteboxTools

Aqui é configurado o motor **WhiteboxTools**, responsável pelo processamento hidrológico e geração dos produtos derivados do DEM:

* `work_dir`: diretório onde o Whitebox irá escrever os arquivos temporários e finais.
* `verbose_mode`: imprime logs detalhados, úteis para auditoria e depuração do processo.
* São definidos nomes de saída padronizados (prefixo + sufixos), facilitando rastreabilidade e organização do pipeline.

**Arquivos intermediários típicos**

* DEM condicionado (`*_dem_filled.tif`)
* Direção de fluxo D8 (`*_d8_pointer.tif`)
* Acumulação D8 (`*_d8_accum.tif`)
* Rede de drenagem / streams (`*_streams.tif`)

**Arquivo final**

* HAND (`*_hand.tif`)

In [ ]:
wbt = WhiteboxTools()
wbt.set_verbose_mode(True)
wbt.work_dir = str(outdir.resolve())

# Arquivos de saída (mesma nomenclatura do script)
dem_filled = str(outdir / f"{prefix}_dem_filled.tif")
flow_dir  = str(outdir / f"{prefix}_d8_pointer.tif")
flow_acc  = str(outdir / f"{prefix}_d8_accum.tif")
streams   = str(outdir / f"{prefix}_streams.tif")
hand      = str(outdir / f"{prefix}_hand.tif")

print("Intermediários/Saídas:")
print(" - DEM condicionado:", dem_filled)
print(" - D8 pointer      :", flow_dir)
print(" - D8 acumulação   :", flow_acc)
print(" - Streams         :", streams)
print(" - HAND            :", hand)

## 6) Execução do pipeline (6 etapas)

A seguir executam-se as operações **na mesma ordem do fluxo original**, transformando o DEM bruto em um produto hidrológico consistente:

> **DEM → DEM condicionado → direção de fluxo (D8) → acumulação → rede de drenagem → HAND**

O objetivo é garantir continuidade hidráulica, extrair os canais e, por fim, estimar a altura relativa ao drenagem mais próximo.


### 6.1 O que é condicionamento do DEM?

Condicionar o DEM significa **pré-processá-lo** para que a água escoe corretamente, removendo artefatos numéricos (píxeis isolados, “poças” falsas, ruído topográfico).

Pode incluir:

* **Preenchimento de depressões** (*fill*), ou
* **Abertura de canais artificiais** (*breach*).

Neste notebook usa-se **apenas uma** dessas opções, controlada por `breach`.

### 6.2 Preencher depressões (*Fill depressions*)

Eleva o fundo das depressões até existir uma saída hidrológica.

* Remove sinks espúrios causados por ruído do DEM.
* Produz uma superfície contínua.

**Vantagem:** simples e robusto.
**Limitação:** pode suavizar excessivamente o relevo.

### 6.3 Romper depressões (*Breach least cost*)

Em vez de elevar o terreno, cria um **canal de menor custo** ligando a depressão à jusante.

* Preserva melhor declividades naturais.
* Evita planícies artificiais.

**Vantagem:** maior realismo geomorfológico.
**Limitação:** pode gerar cortes estreitos se o DEM for ruidoso.

### 6.4 Direção de fluxo D8

Cada célula drena para **um único vizinho** (maior declive entre 8 direções).

Saída: raster de ponteiros indicando o sentido do escoamento.
Na prática, define a “seta” de fluxo em cada píxel.

### 6.5 Acumulação D8

Conta quantas células contribuem a montante.

* valores baixos → encostas
* valores altos → vales/canais

É a base para identificar drenagens.

### 6.6 Extração de drenagens

Aplica-se um **limiar de contribuição**:

* `threshold_cells` = mínimo de células para classificar como canal
* menor limiar → rede densa
* maior limiar → apenas rios principais

A rede resultante é uma aproximação dependente da resolução do DEM.

### 6.7 Cálculo HAND (Height Above Nearest Drainage)

Mede a **altura vertical** de cada célula em relação ao canal hidrologicamente conectado mais próximo.

Interpretação:

* HAND baixo → áreas suscetíveis a inundação
* HAND alto → áreas elevadas/seguras

No WhiteboxTools: `elevation_above_stream(dem, streams)`.

### 6.8 Limpeza

Remove arquivos intermediários (DEM condicionado, direção, acumulação, streams).

* economiza espaço
* reduz rastreabilidade

Controlado pela variável `cleanup`.

In [ ]:
print("[1/6] Condicionando o DEM (preencher ou romper depressões)...")

if breach:
    wbt.breach_depressions_least_cost(dem=str(dem), output=dem_filled)
else:
    wbt.fill_depressions(dem=str(dem), output=dem_filled)

print("[2/6] Calculando direção de fluxo D8 (pointer)...")
wbt.d8_pointer(dem=dem_filled, output=flow_dir)

print("[3/6] Calculando acumulação de fluxo D8 (em células)...")
wbt.d8_flow_accumulation(dem_filled, output=flow_acc, out_type="cells")

print(f"[4/6] Extraindo streams com limiar = {threshold_cells} células...")
wbt.extract_streams(flow_accum=flow_acc, output=streams, threshold=threshold_cells)

print("[5/6] Calculando HAND (Elevation Above Stream)...")
wbt.elevation_above_stream(dem=dem_filled, streams=streams, output=hand)

print(f"[6/6] Concluído! HAND gravado em: {hand}")

if not keep_intermediates:
    for fp in [dem_filled, flow_dir, flow_acc, streams]:
        try:
            os.remove(fp)
        except Exception:
            pass
    print("[i] Intermediários removidos (keep_intermediates=False).")
else:
    print("[i] Intermediários mantidos (keep_intermediates=True).")

## 7) Limiares de alerta / risco de inundação (entrada do usuário)

Em muitos usos operacionais, valores HAND são traduzidos em **níveis de alerta** (por ex., *Alto/Médio/Baixo*).  
Esta célula define explicitamente esses **limiares como variáveis de entrada**, para que o usuário os ajuste.

### Como ler os limiares?
- Suponha que o HAND esteja em **metros**.
- Um exemplo típico (apenas ilustrativo; você deve calibrar localmente):
  - `alto`  : HAND ≤ 5.0 m  
  - `medio` : 1.0 < HAND ≤ 15.0 m  
  - `bajo`  : 3.0 < HAND ≤ 40.0 m  
  - `muy_bajo`: HAND > 40.0 m

> Ajuste esses valores conforme a hidrologia local, a resolução do DEM e (idealmente) evidência histórica (marcas de inundação, modelagem hidráulica ou curvas vazão–nível).



In [ ]:
# Limiares (em metros de HAND). Devem ser crescentes.
hand_thresholds_m = [3, 10, 30]

# Etiquetas (opcional) para reportar/plotar
risk_labels = ["ALTO", "MÉDIO", "BAIXO", "MUITO_BAIXO"]

print("Limiares HAND (m):", hand_thresholds_m)
print("Classes:")
print(" - HAND <= 3 m: ALTO")
print(" - 3 < HAND <= 10 m: MÉDIO")
print(" - 10 < HAND <= 30 m: BAIXO")
print(" - HAND > 30 m: MUITO_BAIXO")


## 8) Mapa de classes de risco (a partir do HAND)

Gera-se um raster categórico de 4 classes (0–3):
- 0: ALTO
- 1: MEDIO
- 2: BAJO
- 3: MUY_BAJO

**Nota:** isto é uma discretização *determinística* baseada em limiares.



In [ ]:
if not Path(hand).exists():
    raise FileNotFoundError(
        f"Raster HAND não encontrado: {hand}. "
        f"Erros externos registrados: {globals().get('external_errors', [])}"
    )

from matplotlib.colors import BoundaryNorm, ListedColormap

t1, t2, t3 = hand_thresholds_m

PROJECT_ROOT = Path(globals().get("PROJECT_ROOT", Path.cwd())).resolve()
outdir = Path(globals().get("BLUMENAU_OUTPUT_DIR", PROJECT_ROOT / "outputs" / "blumenau"))
outdir.mkdir(parents=True, exist_ok=True)

r = (
    rxr.open_rasterio(hand, masked=True)
      .squeeze("band", drop=True)
      .astype("float32")
      .rio.reproject("EPSG:3857")
)

valid = xr.where(np.isfinite(r), True, False)

cls_da = xr.full_like(r, 255, dtype="uint8")
cls_da = cls_da.where(~(valid & (r <= t1)), 0)
cls_da = cls_da.where(~(valid & (r > t1) & (r <= t2)), 1)
cls_da = cls_da.where(~(valid & (r > t2) & (r <= t3)), 2)
cls_da = cls_da.where(~(valid & (r > t3)), 3)

cls_da.rio.write_nodata(255, inplace=True)

class_path = outdir / "blumenau_hand_classes.tif"
cls_da.rio.to_raster(class_path)
print(f"Raster classificado salvo: {class_path}")

class_labels = {
    0: "Alta suscetibilidade",
    1: "Média suscetibilidade",
    2: "Baixa suscetibilidade",
    3: "Muito baixa suscetibilidade",
}

cmap = ListedColormap(["#c62828", "#f9a825", "#2e7d32", "#1565c0"])
norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], cmap.N)

r_plot = cls_da.where(cls_da != 255)

def add_categorical_colorbar(fig, ax, artist):
    cbar = fig.colorbar(artist, ax=ax, ticks=[0, 1, 2, 3], fraction=0.046, pad=0.04)
    cbar.ax.set_yticklabels([
        "0: Alta suscetibilidade",
        "1: Média suscetibilidade",
        "2: Baixa suscetibilidade",
        "3: Muito baixa suscetibilidade",
    ])
    cbar.set_label("Classes de suscetibilidade")
    return cbar

def style_map(ax):
    ax.set_axis_off()
    ax.set_title(
        "Modelo HAND — Blumenau/SC\nMapa de Suscetibilidade a Inundações",
        fontsize=14,
        weight="bold",
    )

fig, ax = plt.subplots(figsize=(10, 10))
img = r_plot.plot.imshow(
    ax=ax,
    cmap=cmap,
    norm=norm,
    add_colorbar=False,
)

try:
    cx.add_basemap(
        ax,
        source=cx.providers.Esri.WorldImagery,
        crs=r_plot.rio.crs,
        attribution_size=6,
    )
except Exception as e:
    print(f"Aviso: basemap remoto indisponível; salvando mapa sem basemap. Detalhe: {type(e).__name__}: {e}")

add_categorical_colorbar(fig, ax, img)
style_map(ax)

output_png = outdir / "mapa_suscetibilidade_blumenau.png"
plt.tight_layout()
plt.savefig(output_png, dpi=300, bbox_inches="tight")
print(f"Mapa PNG salvo: {output_png}")
plt.show()

fig_rel, ax_rel = plt.subplots(figsize=(10, 10))
img_rel = r_plot.plot.imshow(
    ax=ax_rel,
    cmap=cmap,
    norm=norm,
    add_colorbar=False,
)
add_categorical_colorbar(fig_rel, ax_rel, img_rel)
style_map(ax_rel)

output_png_rel = outdir / "mapa_suscetibilidade_blumenau_relatorio.png"
plt.tight_layout()
plt.savefig(output_png_rel, dpi=300, bbox_inches="tight")
print(f"Mapa PNG para relatório salvo: {output_png_rel}")
plt.show()

arr = cls_da.values
arr = arr[arr != 255]
total_pixels = int(arr.size)

print("Percentuais por classe de suscetibilidade:")
for code, label in class_labels.items():
    count = int((arr == code).sum())
    pct = (count / total_pixels * 100.0) if total_pixels else 0.0
    print(f" - {label}: {pct:.2f}% ({count} pixels)")
